In [0]:
from pyspark.sql.functions import current_timestamp

### Extracting data from PostgreSQL(Acting like DB2) using jdbc connection

In [0]:
jdbc_url = "jdbc:postgresql://pg-server-migration.postgres.database.azure.com:5432/migration"
jdbc_user = dbutils.secrets.get(scope="pgDBscope", key="username")
jdbc_password = dbutils.secrets.get(scope="pgDBscope", key="password")
jdbc_driver = "org.postgresql.Driver"

df_raw = spark.read.format("jdbc")\
                .option("url", jdbc_url)\
                .option("user", jdbc_user)\
                .option("password", jdbc_password)\
                .option("driver", jdbc_driver)\
                .option("dbtable", "db2_source.raw_orders")\
                .load()

In [0]:
df_raw.display()

Adding Audit column, timestamp for ingestion time

In [0]:
df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
df_bronze.display()

3. Write to Bronze ADLS Container 
(Auth is handled invisibly by the Unity Catalog External Location)

In [0]:
bronze_path = "abfss://bronze@migrationecom.dfs.core.windows.net/raw_orders"

In [0]:
df_bronze.write.format("delta")\
        .mode("append")\
        .save(bronze_path)